In [0]:
catalog = "transit_analytics"
landing_schema = "landing"
bronze_schema = "bronze"
volume = "raw_gtfs"


def ingest_to_bronze(feed_name: str) -> None:
    """Incrementally ingest raw .pb files for a feed into its Bronze Delta table.

    Args:
        feed_name: Name of the feed folder in the landing volume
            (e.g. 'vehicle_positions', 'trip_updates').
    """
    source_path = f"/Volumes/{catalog}/{landing_schema}/{volume}/{feed_name}"
    checkpoint_path = f"/Volumes/{catalog}/{landing_schema}/{volume}/_checkpoints/{feed_name}"
    target_table = f"{catalog}.{bronze_schema}.{feed_name}"

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .load(source_path)
    )

    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )

In [0]:
ingest_to_bronze("vehicle_positions")
ingest_to_bronze("trip_updates")

In [0]:
print(spark.table("transit_analytics.bronze.vehicle_positions").count())
print(spark.table("transit_analytics.bronze.trip_updates").count())